In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_style("whitegrid")

In [ ]:
survey = pd.read_csv("../data/processed/survey_clean.csv")
survey.head()


In [ ]:
survey.info()
survey.describe()

In [ ]:
numeric_features = survey.select_dtypes(include=['int64','float64'])
numeric_features = numeric_features.fillna(numeric_features.mean())

In [ ]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(numeric_features)

In [ ]:
inertia = []
for k in range(1,10):
    model = KMeans(n_clusters=k, random_state=42)
    model.fit(scaled_data)
    inertia.append(model.inertia_)
plt.plot(range(1,10), inertia, marker='o')
plt.title("Elbow Method for Optimal Clusters")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42)
survey['cluster'] = kmeans.fit_predict(scaled_data)
survey.head()

In [ ]:
sns.countplot(x='cluster', data=survey)
plt.title("Consumer Segment Distribution")
plt.show()

In [ ]:
pca = PCA(n_components=2)
components = pca.fit_transform(scaled_data)
survey['pca1'] = components[:,0]
survey['pca2'] = components[:,1]

In [ ]:
sns.scatterplot(
    x='pca1',
    y='pca2',
    hue='cluster',
    data=survey,
    palette='Set2'
)
plt.title("EV Consumer Segmentation")
plt.show()

In [ ]:
numeric_cols = survey.select_dtypes(include=['int64','float64']).columns
cluster_profile = survey.groupby('cluster')[numeric_cols].mean()
cluster_profile

In [ ]:
cluster_profile.T.plot(figsize=(12,6))
plt.title("Consumer Cluster Characteristics")
plt.xticks(rotation=45, ha='right')   # rotate labels
plt.tight_layout()                    # adjust spacing
plt.show()

In [ ]:
important_features = cluster_profile.columns[:10]
plt.figure(figsize=(12,5))
sns.heatmap(
    cluster_profile[important_features],
    cmap="coolwarm",
    annot=True
)
plt.title("Key Consumer Cluster Characteristics")
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(
    x='pca1',
    y='pca2',
    hue='cluster',
    palette='Set2',
    data=survey,
    s=90,
    edgecolor='black'
)
plt.title("EV Consumer Segments (PCA Projection)", fontsize=14)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Consumer Cluster")
plt.grid(True)
plt.show()

## Random Forest Classification for EV Purchase Prediction

Predicting consumer purchase intent based on survey features using Random Forest Classifier.

In [ ]:
# Create binary target variable: High-value consumers (Cluster 0 = likely buyers, others = less likely)
# This assumes Cluster 0 represents high-intent consumers
target = (survey['cluster'] == 0).astype(int)

survey['purchase_intent'] = target

print(f"Class Distribution:\n{target.value_counts()}")
print(f"\nPercentage of High-Value Consumers: {target.mean()*100:.2f}%")

In [ ]:
# Prepare features and target
X = scaled_data  # Use scaled numeric features
y = target      # Binary purchase intent

# Split data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y  # Ensure balanced class distribution
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")
print(f"Training set class distribution:\n{np.bincount(y_train)}")

In [ ]:
# Train Random Forest Classifier
rf_model = RandomForestClassifier(
    n_estimators=100,      # Number of trees
    max_depth=10,          # Maximum tree depth
    min_samples_split=5,   # Minimum samples to split a node
    random_state=42,
    n_jobs=-1              # Use all available processors
)

rf_model.fit(X_train, y_train)

# Make predictions
y_pred = rf_model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"Random Forest Model Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

In [ ]:
# Detailed model evaluation
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Low Intent', 'High Intent']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

print(f"\nConfusion Matrix:\n{cm}")

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=['Low Intent', 'High Intent'],
    yticklabels=['Low Intent', 'High Intent']
)
plt.title("Confusion Matrix - EV Purchase Intent Prediction")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()

In [ ]:
# Feature Importance Analysis
feature_importance = rf_model.feature_importances_

# Get top 15 most important features
top_indices = np.argsort(feature_importance)[-15:]

feature_names = numeric_features.columns

plt.figure(figsize=(10, 6))
plt.barh(
    range(len(top_indices)), 
    feature_importance[top_indices],
    color='steelblue'
)
plt.yticks(range(len(top_indices)), [feature_names[i] for i in top_indices])
plt.xlabel("Feature Importance Score")
plt.title("Top 15 Most Important Features for EV Purchase Intent")
plt.tight_layout()
plt.show()

# Print top features
print("\nTop 10 Most Important Features:")
for idx, importance in sorted(enumerate(feature_importance), key=lambda x: x[1], reverse=True)[:10]:
    print(f"{feature_names[idx]}: {importance:.4f}")

In [ ]:
# Get prediction probabilities for better insights
y_pred_proba = rf_model.predict_proba(X_test)

# Create results dataframe
results_df = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred,
    'Confidence_Low_Intent': y_pred_proba[:, 0],
    'Confidence_High_Intent': y_pred_proba[:, 1]
})

# Show some sample predictions
print("Sample Predictions with Confidence Scores:")
print(results_df.head(10))

# Model Summary
print("\n=== Random Forest Model Summary ===")
print(f"Total Test Samples: {len(y_test)}")
print(f"Correct Predictions: {(y_test == y_pred).sum()}")
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"Model Parameters:")
print(f"  - Number of trees: 100")
print(f"  - Max depth: 10")
print(f"  - Min samples split: 5")